# 已保存路径的费用拆分（独立诊断）

预先确定：只使用最终 Recorder `300530137db54b6c807c39ebb9e8c331` 的集合与 SOXX 基准已保存持仓。逐日独立重建费用前收益，核对 `net=(1-fee)*gross`，再按原年度和25个训练折汇总。既不重算模型目标，也不重新拟合或模拟其他交易政策。正的同持仓费用前贡献不证明无费用策略可交易，也不是更聪明策略的收益上限。


In [1]:
from pathlib import Path
import hashlib,json
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
ROOT=Path('/Users/rziiiii/dev/workspace-worktrees/constrained-net-excess-20261003')
if not ROOT.exists(): ROOT=Path('/Users/rziiiii/dev/workspace')
EXPERIMENT='constrained_net_excess_20261003'
PARENT_ID='300530137db54b6c807c39ebb9e8c331'
URI='/Users/rziiiii/dev/workspace/research/constrained_net_excess_20261003/qlib_recorder'
qlib.init(provider_uri=str(ROOT/'data/qlib/published_futu_affine_20260925_v1'),region='us',kernels=1)
source_identities=[]
with R.uri_context(URI):
    parent=R.get_recorder(experiment_name=EXPERIMENT,recorder_id=PARENT_ID)
    assert parent.client.get_run(parent.id).info.status=='FINISHED'
    def load_source(name):
        file=Path(parent.get_local_dir())/'artifacts'/name
        digest=hashlib.sha256(file.read_bytes()).hexdigest()
        value=parent.load_object(name)
        assert hashlib.sha256(file.read_bytes()).hexdigest()==digest
        source_identities.append({'artifact':name,'sha256':digest,'bytes':file.stat().st_size})
        return value
    raw=load_source('inputs/primary_frame.pkl')
    replay=load_source('analysis/portfolio_replays.pkl')
    index=load_source('analysis/ensemble_audits.pkl')
    folds=load_source('inputs/development_fold_inputs.pkl')
    folds.update(load_source('inputs/diagnostic_fold_inputs.pkl'))
    comparison=load_source('analysis/comparison.pkl')
    annual_original=load_source('analysis/annual.pkl')
    assert len(index)==len(folds)==25
    training_books=[]
    for row in index:
        identity=(row['variant'],row['year']); fold=folds[identity]
        audit=load_source(row['recorder_artifact'])
        # Saved SOXX book is identical across seeds. Take only benchmark arrays;
        # do not evaluate or retain the seed's candidate/checkpoint trajectory.
        seed_audit=load_source(f"fit_checkpoints/{row['variant']}/{row['year']}/17/training_audit.pkl")
        training_books.append({'variant':row['variant'],'model_year':row['year'],
            'dates':fold['rows'][['signal_date','trade_date','exit_date']].copy(),
            'returns':fold['returns'].copy(),'paths':audit['training_paths'],
            'benchmarks':{c:np.array(v,copy=True) for c,v in seed_audit['benchmark_net_equity'].items()},
            'selected_F':audit['F']})
        del audit,seed_audit
    # Raw frame/paths/selected books are sufficient to reproduce this diagnostic.
    # Parent model objects/checkpoints are not copied or used for inference.
    input_bundle={'raw':raw,'saved_replay':replay,'training_books':training_books,
                  'comparison':comparison,'annual_original':annual_original}
    for identity in source_identities:
        file=Path(parent.get_local_dir())/'artifacts'/identity['artifact']
        assert hashlib.sha256(file.read_bytes()).hexdigest()==identity['sha256']
print({'parent':PARENT_ID,'source_artifacts':len(source_identities),'training_ensembles':len(training_books)})


{'parent': '300530137db54b6c807c39ebb9e8c331', 'source_artifacts': 57, 'training_ensembles': 25}


In [2]:
TOL=1e-12
parity=[]
def decompose(dates,returns,target,net,fee,bench_net,bench_fee,*,variant,cost_bps,scope,model_year=None):
    dates=dates.reset_index(drop=True);r=np.asarray(returns,float);z=np.asarray(target,float)
    net,fee,bnet,bfee=[np.asarray(v,float) for v in (net,fee,bench_net,bench_fee)]
    n=len(dates)
    assert r.shape==(n,2) and z.shape==(n,3)
    assert np.isfinite(r).all() and (r>-1).all() and np.isfinite(z).all()
    assert np.max(np.abs(z.sum(1)-1))<TOL and z.min()>-TOL
    assert all(v.shape==(n,) and np.isfinite(v).all() for v in (net,fee,bnet,bfee))
    assert (net>0).all() and (bnet>0).all() and (fee>=0).all() and (fee<1).all() and (bfee>=0).all() and (bfee<1).all()
    gross=1+z[:,0]*r[:,0]+z[:,1]*r[:,1]
    bgross=1+r[:,1]
    net_error=float(np.max(abs(net-(1-fee)*gross)))
    benchmark_error=float(np.max(abs(bnet-(1-bfee)*bgross)))
    assert net_error<TOL and benchmark_error<TOL
    log_gross=np.log(gross)-np.log(bgross)
    log_fee=np.log1p(-fee)-np.log1p(-bfee)
    log_net=np.log(net)-np.log(bnet)
    error=float(np.max(abs(log_net-log_gross-log_fee)))
    assert error<TOL
    part=dates.copy()
    part['variant']=variant;part['cost_bps']=cost_bps;part['scope']=scope;part['model_year']=model_year
    part['soxl_return']=r[:,0];part['soxx_return']=r[:,1]
    for j,a in enumerate(('soxl','soxx','cash')):part[f'postfee_target_{a}']=z[:,j]
    for name,value in {'net_multiplier':net,'fee_rate':fee,'gross_multiplier':gross,'benchmark_net_multiplier':bnet,'benchmark_fee_rate':bfee,'benchmark_gross_multiplier':bgross,'log_gross_position_excess':log_gross,'log_fee_excess':log_fee,'log_net_excess':log_net}.items():part[name]=value
    parity.append({'variant':variant,'scope':scope,'model_year':model_year,'cost_bps':cost_bps,'rows':n,'net_max_error':net_error,'benchmark_max_error':benchmark_error,'decomposition_max_error':error})
    return part

daily=[]
# Benchmark open_net_weights and learned targets are both normalized after
# funding fees. Verify the saved benchmark is full SOXX rather than assuming
# its fee-rate is a cash allocation or charging the entry twice.
for (segment,variant,cost),path in replay.loc[replay.seed.eq('ensemble')].groupby(['segment','variant','cost_bps']):
    path=path.sort_values('signal_date').reset_index(drop=True)
    ref=replay.loc[replay.segment.eq(segment)&replay.variant.eq('SOXX_buy_hold')&replay.cost_bps.eq(cost)].sort_values('signal_date').reset_index(drop=True)
    assert path[['signal_date','trade_date','exit_date']].equals(ref[['signal_date','trade_date','exit_date']])
    market=raw.loc[pd.DatetimeIndex(path.signal_date)]
    assert np.array_equal(market[['signal_date','trade_date','exit_date']].to_numpy(),path[['signal_date','trade_date','exit_date']].to_numpy())
    assert np.max(abs(ref.soxl_weight))<TOL and np.max(abs(ref.cash_weight))<TOL
    assert np.max(abs(ref.soxx_weight.to_numpy()-1))<TOL
    daily.append(decompose(path[['signal_date','trade_date','exit_date']],market[['soxl_return','soxx_return']].to_numpy(),path[['soxl_weight','soxx_weight','cash_weight']].to_numpy(),1+path.strategy_return.to_numpy(),path.fees.to_numpy(),1+ref.strategy_return.to_numpy(),ref.fees.to_numpy(),variant=variant,cost_bps=int(cost),scope=segment))
for book in training_books:
    n=len(book['dates'])
    for cost,path in book['paths'].items():
        rate=float(cost)/10000
        bfee=np.zeros(n);bfee[0]=rate/(1+rate)
        # One funded entry then unchanged SOXX: validates saved training benchmark
        # without generating any new policy, model target or rebalanced path.
        daily.append(decompose(book['dates'],book['returns'],path['target_weights'],path['net_equity'],path['fees'],book['benchmarks'][cost],bfee,variant=book['variant'],cost_bps=int(cost),scope='training',model_year=int(book['model_year'])))
daily=pd.concat(daily,ignore_index=True)

def agg(part):
    g=float(part.log_gross_position_excess.sum());f=float(part.log_fee_excess.sum());n=float(part.log_net_excess.sum())
    assert abs(n-g-f)<1e-10
    return {'sessions':len(part),'first_signal':str(part.signal_date.min().date()),'first_trade':str(part.trade_date.min().date()),'last_exit':str(part.exit_date.max().date()),'gross_position_log_excess':g,'fee_log_excess':f,'net_log_excess':n,'gross_same_target_relative_wealth':float(np.exp(g)),'fee_relative_multiplier':float(np.exp(f)),'net_relative_wealth':float(np.exp(n)),'gross_annualized_log_excess':252*g/len(part),'fee_annualized_log_excess':252*f/len(part),'net_annualized_log_excess':252*n/len(part),'fee_over_positive_gross':-f/g if g>1e-12 else None}
all_rows=[];annual_rows=[]
for (scope,variant,cost,year),part in daily.groupby(['scope','variant','cost_bps','model_year'],dropna=False):
    row={'scope':scope,'variant':variant,'cost_bps':int(cost),'model_year':None if pd.isna(year) else int(year),**agg(part)}
    all_rows.append(row)
    if scope!='training':
        for signal_year,year_part in part.groupby(part.signal_date.dt.year):annual_rows.append({'scope':scope,'variant':variant,'cost_bps':int(cost),'signal_year':int(signal_year),**agg(year_part)})
aggregate=pd.DataFrame(all_rows);annual=pd.DataFrame(annual_rows)
# Independently recover every saved original full/annual log excess exactly.
for row in comparison.itertuples():
    found=aggregate[(aggregate.scope==row.segment)&(aggregate.variant==row.variant)&(aggregate.cost_bps==row.cost_bps)]
    assert len(found)==1 and abs(found.iloc[0].net_log_excess-row.log_excess_sum)<1e-10
for row in annual_original.itertuples():
    found=annual[(annual.scope==row.segment)&(annual.variant==row.variant)&(annual.cost_bps==row.cost_bps)&(annual.signal_year==row.year)]
    assert len(found)==1 and abs(found.iloc[0].net_log_excess-row.log_excess_sum)<1e-10
paired=[]
for book in training_books:
    for cost in (10,25):
        train=aggregate[(aggregate.scope=='training')&(aggregate.variant==book['variant'])&(aggregate.model_year==book['model_year'])&(aggregate.cost_bps==cost)].iloc[0]
        variant=book['variant'] if book['model_year']<=2022 else book['variant']+'_annual_refit'
        nxt=annual[(annual.variant==variant)&(annual.signal_year==book['model_year'])&(annual.cost_bps==cost)].iloc[0]
        paired.append({'variant':book['variant'],'fit_year':book['model_year'],'cost_bps':cost,'training_sessions':int(train.sessions),'next_signal_year_sessions':int(nxt.sessions),**{'training_'+k:train[k] for k in ('gross_annualized_log_excess','fee_annualized_log_excess','net_annualized_log_excess')},**{'next_year_'+k:nxt[k] for k in ('gross_annualized_log_excess','fee_annualized_log_excess','net_annualized_log_excess')}})
paired=pd.DataFrame(paired)
# Every saved selected training book's mean term recovers its fixed F.
for book in training_books:
    part=aggregate[(aggregate.scope=='training')&(aggregate.variant==book['variant'])&(aggregate.model_year==book['model_year'])]
    assert abs(min(63*part.net_log_excess/part.sessions)-book['selected_F'])<1e-12
print(aggregate[aggregate.scope!='training'].to_string(index=False))
print(annual[(annual.variant=='price_vol')&(annual.cost_bps==25)].to_string(index=False))
print(paired[(paired.variant=='price_vol')&(paired.cost_bps==25)].to_string(index=False))


            scope                         variant  cost_bps  model_year  sessions first_signal first_trade  last_exit  gross_position_log_excess  fee_log_excess  net_log_excess  gross_same_target_relative_wealth  fee_relative_multiplier  net_relative_wealth  gross_annualized_log_excess  fee_annualized_log_excess  net_annualized_log_excess  fee_over_positive_gross
      development                        constant        10         NaN      1762   2016-01-04  2016-01-05 2023-01-04                   0.000000   -4.388850e-19   -4.763519e-14                           1.000000                 1.000000             1.000000                     0.000000              -6.276903e-20              -6.812751e-15                      NaN
      development                        constant        25         NaN      1762   2016-01-04  2016-01-05 2023-01-04                   0.000000   -8.673617e-19    0.000000e+00                           1.000000                 1.000000             1.000000           

In [3]:
def records(frame):return json.loads(frame.to_json(orient='records',date_format='iso',double_precision=15))
source={'parent_experiment':EXPERIMENT,'parent_recorder_id':PARENT_ID,'parent_status':'FINISHED','source_artifacts':source_identities,'no_new_policy_paths':True,'no_fits_or_target_inference':True,'training_benchmark_source':'saved seed17 same-fold benchmark arrays, checked against analytic full-SOXX funded entry','source_prices':'parent primary_frame.pkl from frozen qualified Qlib source, no mutable provider refresh'}
with R.start(experiment_name=EXPERIMENT,uri=URI):
    child=R.get_recorder()
    R.set_tags(role='saved_path_fee_decomposition',parent_recorder_id=PARENT_ID,model_selection_changed=False)
    R.save_objects(artifact_path='inputs',**{'source_identity.pkl':source,'saved_books.pkl':input_bundle})
    R.save_objects(artifact_path='analysis',**{'daily_fee_decomposition.pkl':daily,'aggregate.pkl':aggregate,'annual.pkl':annual,'training_next_year.pkl':paired,'parity.pkl':pd.DataFrame(parity)})
    child_id=child.id
    assert child_id!=PARENT_ID
summary={'experiment':EXPERIMENT,'recorder_id':child_id,'recorder_uri':URI,'parent_recorder_id':PARENT_ID,'diagnostic_only':True,'selection_changed':False,'source_artifact_count':len(source_identities),'training_ensembles':len(training_books),'daily_rows':len(daily),'max_net_reconstruction_error':max(r['net_max_error'] for r in parity),'max_benchmark_reconstruction_error':max(r['benchmark_max_error'] for r in parity),'max_decomposition_error':max(r['decomposition_max_error'] for r in parity),'aggregate':records(aggregate[aggregate.scope!='training']),'annual':records(annual),'training_next_year':records(paired),'limits':['The same saved holdings support this exact accounting identity; removing fees is not a retrained zero-cost strategy or an upper bound on a smarter policy.','No model or portfolio path, threshold, horizon or selection was added. Historical qualification remains false.']}
Path('/tmp/constrained_fee_decomposition_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False))
annual.to_csv('/tmp/constrained_fee_decomposition_annual.csv',index=False)
paired.to_csv('/tmp/constrained_fee_decomposition_training_next_year.csv',index=False)
print({'child_recorder':child_id,'daily_rows':len(daily),'max_net_error':summary['max_net_reconstruction_error'],'max_decomposition_error':summary['max_decomposition_error']})


{'child_recorder': '8fdcc6b190814dd499a4cd2802e2aaa9', 'daily_rows': 125922, 'max_net_error': 4.440892098500626e-16, 'max_decomposition_error': 6.201992180454932e-16}


以下仅把已计算表整理为小摘要并归档，在独立内核执行；不重新计算模型或策略。下载进度组件及git日志已从展示中清除，数值输出与断言完整保留。

In [1]:
from pathlib import Path
import json,os,qlib
from qlib.workflow import R
p=Path('/tmp/constrained_fee_decomposition_summary.json')
full=json.loads(p.read_text())
if 'annual' not in full:full=json.loads(Path('/tmp/constrained_fee_decomposition_metrics.json').read_text())
Path('/tmp/constrained_fee_decomposition_metrics.json').write_text(json.dumps(full,indent=2))
compact={k:v for k,v in full.items() if k not in ('aggregate','annual','training_next_year')}
compact['aggregate']=[{k:r[k] for k in ('scope','variant','cost_bps','sessions','gross_annualized_log_excess','fee_annualized_log_excess','net_annualized_log_excess','net_relative_wealth')} for r in full['aggregate']]
compact['sidecars']=['constrained_fee_decomposition_annual.csv','constrained_fee_decomposition_training_next_year.csv','constrained_fee_decomposition_conclusion.md','constrained_policy_expressivity_review.md']
p.write_text(json.dumps(compact,indent=2))
root=Path('/Users/rziiiii/dev/workspace-worktrees/constrained-net-excess-20261003')
if not root.exists():root=Path('/Users/rziiiii/dev/workspace')
os.chdir(root)
qlib.init(provider_uri=str(root/'data/qlib/published_futu_affine_20260925_v1'),region='us',kernels=1)
with R.start(experiment_name=compact['experiment'],recorder_id=compact['recorder_id'],uri=compact['recorder_uri'],resume=True):
    R.save_objects(artifact_path='analysis',**{'summary.pkl':compact})
print({'summary_bytes':p.stat().st_size,'recorder_id':compact['recorder_id'],'numerical_analysis_changed':False})


{'summary_bytes': 4464, 'recorder_id': '8fdcc6b190814dd499a4cd2802e2aaa9', 'numerical_analysis_changed': False}


# 保存持仓的费用拆分结论

只读取最终主 Recorder300530137db54b6c807c39ebb9e8c331；独立诊断 Recorder8fdcc6b190814dd499a4cd2802e2aaa9保存完整逐日表、源对象与SHA。25个集合训练折及原开发/诊断路径共125,922行；逐日核对 `net=(1-fee)*gross` 最大误差4.44e−16、净对数超额拆分误差6.20e−16；全部原整体/年度超额及25个集合的最差成本F均复原。

开发期2016–2022的年化对数超额拆分：

|模型|费用前持仓贡献|10bp费用项 / 净超额|25bp费用项 / 净超额|
|---|---:|---:|---:|
|price|+2.0749%|−0.7668% / +1.3081%|−1.9171% / +0.1578%|
|price_vol|+2.5445%|−0.8329% / +1.7116%|−2.0823% / +0.4623%|

这里全部是可相加的年化**对数**贡献；不能与普通收益率相加。25bp费用消耗price/price_vol费用前贡献的92.39%/81.83%。price_vol按相同保存持仓，费用前6/7年正，费用后25bp仅2/7年正。2017–2020逐年费用前对数超额+0.310%、+0.831%、+1.641%、+0.613%，相应费用项−1.329%、−1.267%、−1.709%、−1.789%，均翻成负净超额。2021费用前已−0.079%，费用后−3.685%。这些数字支持密集调仓确实消耗了开发期较薄的优势。

训练与下一年也表现出优势衰减。例如用于2017评估年的price_vol模型，此前训练期年化费用前贡献+4.906%，2017评估只+0.311%；用于2021评估的模型训练+8.193%、2021评估−0.079%；用于2023评估的模型训练+7.487%、2023评估−6.249%。所有训练仍有正净值，但下一年不保证。具体25折双成本对照见`constrained_fee_decomposition_training_next_year.csv`。2026是截至09-25开盘的部分年，年化列仅为统一尺度。

费用不是唯一问题。已查看的2023以后诊断，年更模式费用前年化对数超额已经−1.924%，25bp费用项再减−2.740%，净−4.665%；冻结模式费用前−2.585%、费用−3.276%、净−5.861%。年更模式2023、2024、2026费用前皆负，仅2025为正。因此从同一持仓账本中移除全部费用仍不能使这段历史跑赢SOXX。

此分析不产生无费用重训策略、不修改目标或持仓，也不是“更聪明策略的收益上限”。减少交易会改变后续持仓和风险，不能把这里的费用项直接加回并宣称可获得收益。当前模型在固定市场状态下通常只能在单一持仓点精确HOLD，结构复核见`constrained_policy_expressivity_review.md`；那是一个可审查的模型类限制，而不是未来收益证据。

决定不变：三个组仍未取得历史稳定超额资格；没有改动主Recorder、模型选择、旧策略或SIM。下一步如探索表达能力，应先登记单一结构假设，复用已验证会计与真实持仓输入契约，保持成本、风险和年度验证，不用手工交易带修饰现有结果。


# 当前目标表达能力与可复用实现（只读）

当前代码 `src/model/benchmark_anchor_utility.py:99–117` 计算

`target(x,h)=q(x)+alpha_H(x)*h`，

其中 `q=alpha_L*e_L+alpha_C*e_C+(1-alpha_L-alpha_C-alpha_H)*e_S`。输入隐层只依赖标准化市场特征，持仓仅进入最后的 HOLD 专家；三个非负 theta 行的全局总和不超过1。

固定一个市场状态 x，若 `alpha_H<1`，精确不交易的条件只有 `h=q/(1-alpha_H)`。这是持仓单纯形中的一个固定点，不能表示同一市场下围绕目标的一片持仓区域都维持不动。`alpha_H`接近1可以放慢调整，但一般仍产生非零订单；因此每日交易并不直接等于费用训练代码失效。一个特定历史序列也可能恰好走过固定点，这不构成不交易区域的表达能力。

由于基函数在[0,1]，`alpha_j(x)<=sum_k(theta_jk)`。若 `alpha_H=1`，HOLD行就要消耗全部全局预算，SOXL和CASH行必为0。按有限参数下严格介于0和1之间的sigmoid实数定义，HOLD的全部系数须在常数基上，成为处处全HOLD；浮点sigmoid饱和不能恢复已经为0的其他两行。由相同SOXX持仓出发，要在状态x1全仓SOXL、另一个状态x2全现金，分别要求两个行预算至少1，总和至少2，违反总预算≤1。因此不只是优化没走到某些组合，它们本来就在此模型类之外。

这些是结构性事实，不证明更大表达能力会改善真实净超额。全局预算原先提供了有限参数下的精确SOXX和简单可验证合法权重；任何后续表达改变仍需保留这些验收性质，并继续完整费用与原有风险门。

## 仓库已有积累

- `src/model/integrated_allocation.py:126–150` 的 `IntegratedAllocationPolicy` 已将真实盘前3资产持仓与市场特征拼接输入网络，`rollout_policy`使用自己费用后、收益标记后的状态；测试中已有状态时序、费用资金与独立集合持仓验证。可以复用状态契约及核对，而不是重新造会计。该版本输出softmax，有限logits不能达到精确边界或自动提供HOLD区域；Phase1 `stable_excess_integration_20261002` 历史门失败，且基于后来发现有价格口径问题的旧源，不能把原高收益当作有效alpha证据。
- `src/workflow/pit_constituent_auxiliary_training_allocation.py:973` 的 `_holdings_aware_turnover` 已修复“目标差值代替真实订单”的问题。旧卡 `holdings_aware_pit_allocation_v1` Recorder77a5c2bb26f34fe0bf6e19d45f15a2a5，在2016–2022、10bp选择诊断候选，交易602/1762天、MaxDD−57.69%，未晋升。此处“持仓感知”主要指换手会计，并不等于网络读实际持仓。它仍采用EWM及rebalance_threshold（同文件1312–1338），以及soft_trade_event惩罚（872行）；不能把减少交易证明成无阈值的学得HOLD区域，也不能直接复用旧的近似扣费 `gross - cost*turnover` 取代本轮精确扣费。
- `src/model/benchmark_anchor_utility.py:120–136` 已有非负L1球投影及其测试，可复用投影数学/可行性验收，但它当前投影的是全局参数，不是状态相关的动作单纯形激活。
- 已对当前工作树及 `git grep HEAD -- src tests` 检查 `sparsemax|entmax`，未找到现成实现。名称含`sparse_execution`的 `src/workflow/soxl_compounding_regime_sparse_execution.py` 是固定2%/5% no-trade band、最短持有等旧规则实验（42–43、850–867行），不是稀疏概率映射；其结论明确提示固定band/dwell会丢弃有效日频目标变化。

本次只做结构证明与复用盘点，没有实现替代模型、交易带、阈值或新增拟合。费用诊断也显示近期保存持仓在扣费前已经落后；增强HOLD表达能力不能被预先当成充分修复。
